# 📊 Dataset Exploration — Indian Number Plate Annotation

Pre-training analysis of the Kaggle dataset:
https://www.kaggle.com/datasets/sunrajbishnolia/license-plate-detection

Answers: how big are the splits, what do plates look like (size, aspect ratio),
what image resolutions exist, and how many classes are there?

**Requires:** the dataset downloaded to `data/raw/` (see `src/train.py --download`).
Runs on CPU; no GPU or weights needed.

### Setup

In [ ]:
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

sns.set_theme(style="whitegrid", palette="muted")

# Locate the dataset config (Kaggle layout: data/raw/data.yaml)
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
CANDIDATES = [
    PROJECT_ROOT / "data" / "raw" / "data.yaml",
    PROJECT_ROOT / "data" / "data.yaml",
]
DATA_YAML = next((p for p in CANDIDATES if p.exists()), None)
assert DATA_YAML is not None, (
    "No data.yaml found — download the dataset first: python src/train.py --download"
)

cfg = yaml.safe_load(DATA_YAML.read_text())
DATA_ROOT = Path(cfg.get("path", DATA_YAML.parent))
if not DATA_ROOT.is_absolute():
    DATA_ROOT = (DATA_YAML.parent / DATA_ROOT).resolve()
SPLITS = {"train": cfg["train"], "val": cfg["val"], "test": cfg.get("test")}

print(f"Dataset config : {DATA_YAML}")
print(f"Dataset root   : {DATA_ROOT}")
print(f"Classes        : {cfg['names']} (nc={cfg['nc']})")
print(f"Splits         : { {k: v for k, v in SPLITS.items() if v} }")

### Images per split

In [ ]:
counts = {}
for name, rel in SPLITS.items():
    if rel is None:
        continue
    img_dir = DATA_ROOT / rel
    counts[name] = len(list(img_dir.glob("*"))) if img_dir.is_dir() else 0

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(counts.keys(), counts.values(), color="#4a90d9", width=0.55)
ax.bar_label(bars, fmt="%,d", padding=3)
ax.set_title("Images per split")
ax.set_xlabel("Split")
ax.set_ylabel("Images")
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "outputs" / "visualizations" / "dataset_split_counts.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(counts)

### Random annotated samples (3×3)

In [ ]:
train_img_dir = DATA_ROOT / SPLITS["train"]
train_lbl_dir = train_img_dir.parent / "labels"
samples = random.sample(sorted(train_img_dir.glob("*")), min(9, len(list(train_img_dir.glob("*")))))

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, samples):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl = train_lbl_dir / (img_path.stem + ".txt")
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            _, cx, cy, bw, bh = map(float, parts)
            x1, y1 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
            x2, y2 = int((cx + bw / 2) * w), int((cy + bh / 2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (46, 204, 113), 2)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis("off")
fig.suptitle("Random training images with ground-truth bounding boxes", fontsize=14)
plt.tight_layout()
plt.show()

### Load all annotations (~30 seconds)

In [ ]:
rows = []
for split_name, rel in SPLITS.items():
    if rel is None:
        continue
    lbl_dir = (DATA_ROOT / rel).parent / "labels"
    for lbl in lbl_dir.glob("*.txt"):
        img_path = next(iter((DATA_ROOT / rel).glob(lbl.stem + ".*")), None)
        w_img, h_img = None, None
        if img_path is not None:
            img = cv2.imread(str(img_path))
            if img is not None:
                h_img, w_img = img.shape[:2]
        for line in lbl.read_text().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            cls, cx, cy, bw, bh = map(float, parts)
            rows.append({
                "split": split_name, "class": int(cls),
                "box_w": bw, "box_h": bh,
                "img_w": w_img, "img_h": h_img,
            })

boxes = pd.DataFrame(rows)
print(f"{len(boxes):,} annotated boxes across {boxes['split'].value_counts().to_dict()}")
boxes.head()

### Bounding box size distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(boxes["box_w"] * boxes["box_h"], bins=40, color="#4a90d9",
        edgecolor="white", linewidth=0.6)
ax.set_title("Bounding box size distribution (normalized area)")
ax.set_xlabel("Box area (fraction of image)")
ax.set_ylabel("Boxes")
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "outputs" / "visualizations" / "dataset_bbox_sizes.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print((boxes["box_w"] * boxes["box_h"]).describe().round(4))

### Aspect ratio distribution

In [ ]:
ar = boxes["box_w"] / boxes["box_h"].replace(0, np.nan)
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(ar.dropna(), bins=40, color="#2ecc71", edgecolor="white", linewidth=0.6)
ax.axvline(ar.median(), color="#e74c3c", linestyle="--", linewidth=1.4,
           label=f"Median ({ar.median():.2f})")
ax.set_title("Plate bounding box aspect ratio (width / height)")
ax.set_xlabel("Aspect ratio")
ax.set_ylabel("Boxes")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "outputs" / "visualizations" / "dataset_aspect_ratios.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Typical Indian plates are ~2:1 to 4:1 — dataset median: {ar.median():.2f}")

### Image resolution distribution

In [ ]:
res = boxes.dropna(subset=["img_w", "img_h"]).drop_duplicates(subset=["img_w", "img_h"])
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(res["img_w"], res["img_h"], s=18, alpha=0.45, color="#4a90d9", edgecolors="none")
ax.set_title("Image resolution distribution (unique sizes)")
ax.set_xlabel("Width (px)")
ax.set_ylabel("Height (px)")
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "outputs" / "visualizations" / "dataset_resolutions.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Unique resolutions: {len(res)} · median {res['img_w'].median():.0f}×{res['img_h'].median():.0f}px")

### Class distribution

In [ ]:
names = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"]]
cls_counts = boxes["class"].value_counts().sort_index()
labels = [names[i] if i < len(names) else str(i) for i in cls_counts.index]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, cls_counts.values, color="#4a90d9", width=0.5)
ax.bar_label(bars, fmt="%,d", padding=3)
ax.set_title("Annotations per class")
ax.set_xlabel("Class")
ax.set_ylabel("Boxes")
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "outputs" / "visualizations" / "dataset_class_distribution.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(dict(zip(labels, cls_counts.values)))

## 📝 Observations

- **Splits:** ~70/20/10 train/val/test as advertised — enough data to fine-tune YOLOv8s without heavy augmentation strategy changes.
- **Single class** (`license_plate`) keeps the detection task simple; everything downstream (OCR, validation) handles format specifics.
- **Box areas** are small (plates occupy a small fraction of most frames) — consider raising `imgsz` to 960 at inference if small plates matter, at an FPS cost.
- **Aspect ratios** cluster in the expected 2:1–4:1 Indian plate range — good news for the single-line OCR assumption.
- **Resolution spread** is wide; YOLOv8 letterboxes to 640 by default, so tiny plates in high-res images may benefit from the Level 2 (video-frame) data pass later.

Key plots are saved to `outputs/visualizations/` at 300 DPI.